# 강남구 자전거 재배치 쌍 이동 검증

**목적**: 데이터에서 재배치 의심 시간대를 추출하고, 트럭 재배치(A→B 쌍 이동)인지 검증

**데이터**: 서울특별시 대여소별 공공자전거 대여가능 수량(1시간 단위)
**기간**: 2022년 1월 ~ 2023년 3월 (15개월)

## 1. 환경 설정 및 데이터 경로

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("/Users/cheng80/Desktop/ddri_work")
DATA_DIR = ROOT / "3조 공유폴더" / "서울특별시_대여소별 공공자전거 대여가능 수량(1시간 단위)_20230331"
MAPPING_PATH = ROOT / "cheng80" / "api_output" / "ddri_full161_station_api_mapping_table.csv"
OUTPUT_DIR = ROOT / "ddri_rebalance_verification" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_INCREASE = 8
THRESHOLD_DECREASE = -8
# 출퇴근 시간대: 재배치 후보에서 제외 (7~10시 출근, 15~20시 퇴근·오후피크)
COMMUTE_HOURS = {7, 8, 9, 10, 15, 16, 17, 18, 19, 20}

print(f"데이터: {DATA_DIR}")
print(f"출력: {OUTPUT_DIR}")

데이터: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/서울특별시_대여소별 공공자전거 대여가능 수량(1시간 단위)_20230331
출력: /Users/cheng80/Desktop/ddri_work/ddri_rebalance_verification/output


## 2. 강남구 데이터 로드

In [2]:
def load_gangnam_station_ids():
    mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
    return set(mapping["station_id"].astype(int))

def load_monthly_data():
    gangnam_ids = load_gangnam_station_ids()
    parts = []
    for m in range(1, 13):
        fname = f"22.{m:02d}.csv"
        path = DATA_DIR / fname
        if path.exists():
            df = pd.read_csv(path, encoding="cp949")
            df = df.rename(columns={"일시": "date", "대여소번호": "station_id", "대여소명": "station_name", "시간대": "hour", "거치대수량": "stock"})
            df["station_id"] = pd.to_numeric(df["station_id"], errors="coerce")
            df = df.dropna(subset=["station_id"])
            df["station_id"] = df["station_id"].astype(int)
            df = df[df["station_id"].isin(gangnam_ids)]
            parts.append(df)
    for fname in ["23.01.csv", "23.02.csv", "23.03.csv"]:
        path = DATA_DIR / fname
        if path.exists():
            df = pd.read_csv(path, encoding="cp949")
            df = df.rename(columns={"일시": "date", "대여소번호": "station_id", "대여소명": "station_name", "시간대": "hour", "거치대수량": "stock"})
            df["station_id"] = pd.to_numeric(df["station_id"], errors="coerce")
            df = df.dropna(subset=["station_id"])
            df["station_id"] = df["station_id"].astype(int)
            df = df[df["station_id"].isin(gangnam_ids)]
            parts.append(df)
    return pd.concat(parts, ignore_index=True)

df = load_monthly_data()
dt = pd.to_datetime(df["date"], errors="coerce")
print(f"강남구 행 수: {len(df):,}")
print(f"기간: {dt.min().date()} ~ {dt.max().date()}")

강남구 행 수: 820,978
기간: 2022-01-01 ~ 2023-03-16


## 3. 시간대별 변화량(delta) 계산 및 급격한 증감 탐지

In [3]:
df = df.sort_values(["station_id", "date", "hour"])
df["prev_stock"] = df.groupby("station_id")["stock"].shift(1)
df["delta"] = df["stock"] - df["prev_stock"]
df = df.dropna(subset=["delta"])

df["large_increase"] = df["delta"] >= THRESHOLD_INCREASE
df["large_decrease"] = df["delta"] <= THRESHOLD_DECREASE
large_events = df[df["large_increase"] | df["large_decrease"]].copy()

print("급격한 증감(|delta|>=8) 탐지 완료")

급격한 증감(|delta|>=8) 탐지 완료


## 4. 시간대별 집계 및 데이터 기반 재배치 시간대 추출

In [4]:
inc = large_events[large_events["large_increase"]].groupby("hour")["delta"].agg(["mean", "count", "median"])
dec = large_events[large_events["large_decrease"]].groupby("hour")["delta"].agg(["mean", "count", "median"])
inc.columns = ["inc_mean", "inc_count", "inc_median"]
dec.columns = ["dec_mean", "dec_count", "dec_median"]

by_hour = df.groupby("hour").agg(
    large_increase=("large_increase", "sum"),
    large_decrease=("large_decrease", "sum"),
).reset_index()
by_hour["total"] = by_hour["large_increase"] + by_hour["large_decrease"]
by_hour = by_hour.merge(inc, on="hour", how="left").merge(dec, on="hour", how="left")

by_station = df.groupby("station_id").agg(
    station_name=("station_name", "first"),
    large_increase=("large_increase", "sum"),
    large_decrease=("large_decrease", "sum"),
).reset_index()
by_station["total"] = by_station["large_increase"] + by_station["large_decrease"]
by_station = by_station.sort_values("total", ascending=False).head(20)

# 데이터 기반 재배치 시간대 추출: 급증≈급감(ratio 0.7~1.3), 출퇴근 시간대 제외
by_hour["ratio"] = by_hour["large_increase"] / by_hour["large_decrease"].clip(lower=1)
mask = (
    (by_hour["ratio"] >= 0.7) & (by_hour["ratio"] <= 1.3)
    & (by_hour["total"] >= 30)
    & (~by_hour["hour"].isin(COMMUTE_HOURS))
)
rebalance_hours = sorted(by_hour[mask]["hour"].tolist())
print(f"데이터에서 추출한 재배치 의심 시간대: {rebalance_hours} (급증/급감 0.7~1.3, 이벤트≥30, 출퇴근 제외)")

by_hour.to_csv(OUTPUT_DIR / "ddri_rebalance_by_hour_2022_01_2023_03.csv", index=False, encoding="utf-8-sig")
by_station.to_csv(OUTPUT_DIR / "ddri_rebalance_by_station_top20_2022_01_2023_03.csv", index=False, encoding="utf-8-sig")

display(by_hour[["hour", "large_increase", "large_decrease", "total", "ratio"]].head(12))

,hour,large_increase,large_decrease,total
0,0,230,210,440
1,1,5,8,13
2,2,1,7,8
3,3,1,1,2
4,4,0,0,0
5,5,1,1,2
6,6,5,0,5
7,7,17,1,18
8,8,275,65,340
9,9,999,295,1294


## 5. 재배치 시간대 일별 쌍 이동 검증 (balance_ratio)

In [5]:
def sum_positive(d):
    return d[d > 0].sum()

def sum_negative(d):
    return d[d < 0].sum()

rebal_events = large_events[
    (large_events["hour"].isin(rebalance_hours)) & (large_events["date"].notna())
].copy()
rebal_events["date"] = pd.to_datetime(rebal_events["date"])

by_date = rebal_events.groupby("date").agg(
    total_increase=("delta", sum_positive),
    total_decrease=("delta", sum_negative),
    n_increase=("large_increase", "sum"),
    n_decrease=("large_decrease", "sum"),
).reset_index()
by_date["abs_decrease"] = by_date["total_decrease"].abs()
by_date["balance_ratio"] = by_date["total_increase"] / (by_date["abs_decrease"] + 1e-9)
by_date["imbalance"] = (by_date["total_increase"] + by_date["total_decrease"]).abs()

by_date.to_csv(OUTPUT_DIR / "ddri_rebalance_0h_by_date_2022_01_2023_03.csv", index=False, encoding="utf-8-sig")

balanced = by_date[(by_date["balance_ratio"] >= 0.5) & (by_date["balance_ratio"] <= 2.0)]
valid_0 = by_date[(by_date["abs_decrease"] >= 50) & (by_date["total_increase"] >= 50)]
print(f"쌍 이동 가능(0.5~2.0): {len(balanced)}일 / {len(by_date)}일")
print(f"유효일 balance_ratio 평균: {valid_0['balance_ratio'].mean():.2f}" if len(valid_0) > 0 else "유효일 없음")
display(by_date.head(10))

쌍 이동 가능(0.5~2.0): 7일 / 25일
유효일 balance_ratio 평균: 1.47


,date,total_increase,total_decrease,n_increase,n_decrease,abs_decrease,balance_ratio,imbalance
0,2022-02-01,162.0,-58.0,17,6,58.0,2.793103e+00,104.0
1,2022-03-01,127.0,-215.0,12,20,215.0,5.906977e-01,88.0
2,2022-04-04,378.0,-180.0,24,16,180.0,2.100000e+00,198.0
3,2022-04-16,0.0,-8.0,0,1,8.0,0.000000e+00,8.0
4,2022-05-01,149.0,-125.0,12,11,125.0,1.192000e+00,24.0
5,2022-05-05,0.0,-13.0,0,1,13.0,0.000000e+00,13.0
6,2022-05-07,9.0,0.0,1,0,0.0,9.000000e+09,9.0
7,2022-05-10,8.0,0.0,1,0,0.0,8.000000e+09,8.0
8,2022-06-01,79.0,-75.0,8,7,75.0,1.053333e+00,4.0
9,2022-06-04,9.0,0.0,1,0,0.0,9.000000e+09,9.0


## 6. 재배치 시간대 선정 근거 차트 (데이터 기반 추출 결과)

In [6]:
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

rebalance_hours_set = set(rebalance_hours)
colors = ["#1565c0" if h in rebalance_hours_set else "#90a4ae" if h in [9, 15] else "#b0bec5" for h in by_hour["hour"]]
axes[0].bar(by_hour["hour"], by_hour["total"], color=colors, edgecolor="white")
for h in rebalance_hours_set:
    axes[0].axvspan(h - 0.5, h + 0.5, alpha=0.15, color="#1565c0")
axes[0].set_xlabel("시간대")
axes[0].set_ylabel("급격한 증감 횟수")
axes[0].set_title("시간대별 급격한 증감 횟수\n(파란색: 데이터 추출 재배치 시간대, 회색: 출퇴근 피크)")
axes[0].set_xticks(by_hour["hour"])
for h in rebalance_hours_set:
    row = by_hour[by_hour["hour"] == h]
    if len(row) > 0:
        axes[0].text(h, row["total"].values[0] + 15, f"{h}시", ha="center", fontsize=8, color="#1565c0")
axes[0].grid(True, alpha=0.3, axis="y")

ratio = by_hour["large_increase"] / by_hour["large_decrease"].clip(lower=1)
ratio = ratio.clip(upper=5)
axes[1].bar(by_hour["hour"], ratio, color=colors, edgecolor="white")
axes[1].axhline(1.0, color="#2e7d32", linestyle="--", linewidth=1.5, label="1:1 균형 (쌍 이동)")
axes[1].axhspan(0.7, 1.3, alpha=0.1, color="green")
axes[1].set_xlabel("시간대")
axes[1].set_ylabel("급증/급감 비율")
axes[1].set_title("시간대별 급증/급감 비율\n(비율≈1: 재배치 의심, 비율>2: 사용자 출퇴근)")
axes[1].set_xticks(by_hour["hour"])
axes[1].set_ylim(0, 5)
axes[1].legend(loc="upper right", fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ddri_rebalance_0h_22h_23h_selection_evidence.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {OUTPUT_DIR / 'ddri_rebalance_0h_22h_23h_selection_evidence.png'}")

저장: /Users/cheng80/Desktop/ddri_work/ddri_rebalance_verification/output/ddri_rebalance_0h_selection_evidence.png


## 7. 쌍 이동 검증 스캐터 차트

In [7]:
df_chart = by_date.copy()
df_chart["date"] = pd.to_datetime(df_chart["date"])
valid = df_chart[(df_chart["abs_decrease"] >= 20) & (df_chart["total_increase"] >= 20) & (df_chart["balance_ratio"] < 10)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

paired = valid[(valid["balance_ratio"] >= 0.5) & (valid["balance_ratio"] <= 2.0)]
other = valid[(valid["balance_ratio"] < 0.5) | (valid["balance_ratio"] > 2.0)]

axes[0].scatter(other["abs_decrease"], other["total_increase"], s=80, alpha=0.7, c="#999999", label="쌍 이동 아님")
axes[0].scatter(paired["abs_decrease"], paired["total_increase"], s=100, alpha=0.8, c="#2e7d32", label="쌍 이동 가능 (0.5~2.0)")
lim_max = max(valid["abs_decrease"].max(), valid["total_increase"].max()) * 1.05
axes[0].plot([0, lim_max], [0, lim_max], "k--", linewidth=1.5, alpha=0.7, label="y=x (급증합=급감합)")
for _, row in paired.iterrows():
    axes[0].annotate(row["date"].strftime("%m/%d"), (row["abs_decrease"], row["total_increase"]), fontsize=8, xytext=(5, 5), textcoords="offset points")
axes[0].set_xlabel("급감합 (절대값, 대)")
axes[0].set_ylabel("급증합 (대)")
axes[0].set_title("재배치 시간대 일별 급증합 vs 급감합\n(y=x에 가까우면 트럭 A→B 쌍 이동)")
axes[0].legend(loc="upper left", fontsize=9)
axes[0].set_xlim(0, lim_max)
axes[0].set_ylim(0, lim_max)
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

all_valid = df_chart[(df_chart["balance_ratio"] > 0) & (df_chart["balance_ratio"] < 10)]
all_valid["paired"] = (all_valid["balance_ratio"] >= 0.5) & (all_valid["balance_ratio"] <= 2.0)
colors = ["#2e7d32" if p else "#999999" for p in all_valid["paired"]]
axes[1].scatter(all_valid["date"], all_valid["balance_ratio"], s=70, c=colors, alpha=0.8, edgecolor="white", linewidth=0.5)
axes[1].axhline(1.0, color="#1565c0", linestyle="-", linewidth=1.5, label="balance=1 (완전 쌍 이동)")
axes[1].axhspan(0.5, 2.0, alpha=0.12, color="green", label="쌍 이동 가능 구간 (0.5~2.0)")
axes[1].set_xlabel("날짜")
axes[1].set_ylabel("balance_ratio (급증합/급감합)")
axes[1].set_title("재배치 시간대 일별 balance_ratio\n(1에 가까울수록 쌍 이동)")
axes[1].legend(loc="upper right", fontsize=9)
axes[1].set_ylim(0, 3.5)
axes[1].grid(True, alpha=0.3)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ddri_rebalance_verification_scatter.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {OUTPUT_DIR / 'ddri_rebalance_verification_scatter.png'}")

저장: /Users/cheng80/Desktop/ddri_work/ddri_rebalance_verification/output/ddri_rebalance_verification_scatter.png


/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_40801/1094974721.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_valid["paired"] = (all_valid["balance_ratio"] >= 0.5) & (all_valid["balance_ratio"] <= 2.0)


## 8. 0·22·23시 재배치 월별 날짜 히트맵

In [8]:
# 월별 x 일(1~31) 히트맵: 0·22·23시 재배치가 일어난 날 표시
rebal_events = large_events[large_events["hour"].isin(rebalance_hours)].copy()
rebal_events["date"] = pd.to_datetime(rebal_events["date"])
rebal_dates = rebal_events.groupby(rebal_events["date"].dt.date).size().reset_index(name="n")
rebal_dates["date"] = pd.to_datetime(rebal_dates["date"])
rebal_dates["month"] = rebal_dates["date"].dt.to_period("M")
rebal_dates["day"] = rebal_dates["date"].dt.day

months = sorted(rebal_dates["month"].unique())
heatmap_data = []
for m in months:
    row = [0] * 31
    days_in_m = rebal_dates[rebal_dates["month"] == m]["day"].tolist()
    for d in days_in_m:
        row[d - 1] = 1
    heatmap_data.append(row)

import numpy as np
arr = np.array(heatmap_data)
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(arr, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(len(months)))
ax.set_yticklabels([str(m) for m in months], fontsize=9)
ax.set_xticks(range(0, 31, 2))
ax.set_xticklabels(range(1, 32, 2))
ax.set_xlabel("일 (1~31)")
ax.set_ylabel("월")
ax.set_title("재배치 발생 날짜 (월별)\n색: 해당 일에 재배치 이벤트 있음")
ax.set_xlim(-0.5, 30.5)
plt.colorbar(im, ax=ax, label="이벤트 있음(1)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ddri_rebalance_0h_22h_23h_by_month_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {OUTPUT_DIR / 'ddri_rebalance_0h_22h_23h_by_month_heatmap.png'}")

저장: /Users/cheng80/Desktop/ddri_work/ddri_rebalance_verification/output/ddri_rebalance_0h_by_month_heatmap.png


## 9. 재배치 스테이션 분포 (골고루 vs 랜덤)

In [ ]:
# 0·22·23시 재배치만 필터
rebal = large_events[large_events["hour"].isin(rebalance_hours)]
by_st_rebal = rebal.groupby("station_id").size().reset_index(name="count")
st_names = df.groupby("station_id")["station_name"].first().reset_index()
by_st_rebal = by_st_rebal.merge(st_names, on="station_id", how="left")
by_st_rebal = by_st_rebal.sort_values("count", ascending=False)

n_stations = len(by_st_rebal)
total_events = by_st_rebal["count"].sum()
top10_share = by_st_rebal.head(10)["count"].sum() / total_events * 100
top20_share = by_st_rebal.head(20)["count"].sum() / total_events * 100
p = by_st_rebal["count"] / total_events
hhi = (p ** 2).sum() * 10000

print(f"=== 재배치(시간대 {rebalance_hours}) 스테이션별 분포 ===")
print(f"이벤트 있는 스테이션: {n_stations} / 161")
print(f"총 이벤트: {total_events}")
print(f"상위 10개 점유율: {top10_share:.1f}%")
print(f"상위 20개 점유율: {top20_share:.1f}%")
print(f"HHI(집중도): {hhi:.0f}")
print(f"재배치 0건 스테이션: {161 - n_stations}개")
display(by_st_rebal[["station_id", "station_name", "count"]].head(10))

by_st_rebal.to_csv(OUTPUT_DIR / "ddri_rebalance_0h_22h_23h_by_station.csv", index=False, encoding="utf-8-sig")

# 스테이션별 빈도 차트: 상위 20개 가로 막대 + 이벤트 수 분포 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top20 = by_st_rebal.head(20)
def short_label(row):
    s = str(row["station_name"]).split(". ", 1)[-1] if ". " in str(row["station_name"]) else str(row["station_name"])
    return f"{row['station_id']} {s[:10]}…" if len(s) > 10 else f"{row['station_id']} {s}"
labels = [short_label(row) for _, row in top20.iterrows()]
axes[0].barh(range(len(top20)), top20["count"].values, color="#1565c0", alpha=0.85, edgecolor="white")
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(labels, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("재배치 이벤트 수")
axes[0].set_title("상위 20개 스테이션 재배치 빈도")
axes[0].grid(True, alpha=0.3, axis="x")

axes[1].hist(by_st_rebal["count"], bins=range(1, int(by_st_rebal["count"].max()) + 2), color="#1565c0", alpha=0.85, edgecolor="white")
axes[1].set_xlabel("재배치 이벤트 수")
axes[1].set_ylabel("스테이션 수")
axes[1].set_title("이벤트 수 분포 (골고루 정도)\n대부분 1~10회 구간에 분포")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ddri_rebalance_station_frequency.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"저장: {OUTPUT_DIR / 'ddri_rebalance_station_frequency.png'}")

## 10. 완료

In [9]:
print("산출물:")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        print(f"  - {f.name}")

산출물:
  - ddri_rebalance_0h_by_date_2022_01_2023_03.csv
  - ddri_rebalance_0h_by_month_heatmap.png
  - ddri_rebalance_0h_selection_evidence.png
  - ddri_rebalance_by_hour_2022_01_2023_03.csv
  - ddri_rebalance_by_station_top20_2022_01_2023_03.csv
  - ddri_rebalance_verification_scatter.png
